# Preparación de las series temporales

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

Configuraciones iniciales:

In [ ]:
PROJECT_ROOT = Path()

METADATA_PATH = PROJECT_ROOT / "metadata_filtered.csv"
RAW_DIR = PROJECT_ROOT / "raw" / "raw_pub"
OUTPUT_DIR = PROJECT_ROOT / "processed"

OUTPUT_PARQUET = OUTPUT_DIR / "consumption_all_users.parquet"
QUALITY_CSV = OUTPUT_DIR / "quality_report.csv"
QUALITY_PARQUET = OUTPUT_DIR / "quality_report.parquet"
METADATA_PARQUET = OUTPUT_DIR / "metadata_filtered.parquet"

# Nombre de la columna que contiene el ID en metadata_filtered.csv
USER_COLUMN = "user"

# Configuración del CSV crudo
RAW_SEPARATOR = ","
RAW_COLUMN_NAMES = ["Fecha", "consumo"]

# Frecuencia esperada
EXPECTED_FREQ = "h"

# Parquet
PARQUET_COMPRESSION = "zstd"
OVERWRITE = True

# Seguir aunque un user de error
CONTINUE_ON_USER_ERROR = True

# Los valores negativos los asumimos como valores perdidos
TREAT_NEGATIVE_AS_MISSING = True

Cargar usuarios seleccionados en el notebook anterior:

In [ ]:
metadata = pd.read_csv(METADATA_PATH)

unnamed_cols = [c for c in metadata.columns if c.startswith("Unnamed:")]
if unnamed_cols:
    metadata = metadata.drop(columns=unnamed_cols)

if USER_COLUMN not in metadata.columns:
    raise ValueError(
        f"No existe la columna {USER_COLUMN!r} en {METADATA_PATH}. "
        f"Columnas disponibles: {metadata.columns.tolist()}"
    )

metadata[USER_COLUMN] = metadata[USER_COLUMN].astype(str)

if metadata[USER_COLUMN].duplicated().any():
    duplicated = metadata.loc[metadata[USER_COLUMN].duplicated(keep=False), USER_COLUMN].unique()

    raise ValueError(
        "Hay IDs duplicados en metadata_filtered.csv. "
        f"Ejemplos: {duplicated[:10].tolist()}")

all_user_ids = metadata[USER_COLUMN].tolist()
user_ids = all_user_ids

print(f"Usuarios en metadata_filtered.csv: {len(all_user_ids):,}")
print(f"Usuarios que se procesarán en esta ejecución: {len(user_ids):,}")
print(f"Carpeta raw: {RAW_DIR}")

if not RAW_DIR.exists():
    raise FileNotFoundError(f"No existe la carpeta: {RAW_DIR}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata.rename(columns={USER_COLUMN: "user_id"}).to_parquet(
    METADATA_PARQUET,
    index=False,
    compression=PARQUET_COMPRESSION,
)

display(metadata.head())

Usuarios en metadata_filtered.csv: 1,025
Usuarios que se procesarán en esta ejecución: 1,025
Carpeta raw: C:\Mis documentos\Master\TFM\raw\raw_pub


,user,start_date,end_date,length_days,length_years,potential_samples,actual_samples,missing_samples_abs,missing_samples_pct,contract_start_date,contract_end_date,contracted_tariff,self_consumption_type,p1,p2,p3,p4,p5,p6,province,municipality,zip_code,cnae
0,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31T01:00:00Z,2022-06-05T00:00:00Z,1831.0,5.013005,43944,43824,120,0.273075,2021-06-01,NaN,2.0TD,NaN,4.40,4.40,NaN,NaN,NaN,NaN,Bizkaia,Bilbao,48012.0,9820.0
1,0062c8d5ca7dfed574aab968f665978a9cb770a0d8f325...,2018-12-05T01:00:00Z,2022-06-05T00:00:00Z,1278.0,3.498973,30672,30665,7,0.022822,2021-06-01,NaN,2.0TD,NaN,2.20,2.20,NaN,NaN,NaN,NaN,Bizkaia,Bilbao,48006.0,9820.0
2,00b655f77a0bf8c12323560fa1cd609548ab10782e495b...,2018-09-21T01:00:00Z,2022-06-05T00:00:00Z,1353.0,3.704312,32472,32464,8,0.024637,2021-06-01,NaN,2.0TD,NaN,3.30,3.30,NaN,NaN,NaN,NaN,Bizkaia,Bilbao,48006.0,9820.0
3,00db2f868b1075624b10684c1bbade33552a0490b27e01...,2017-05-31T01:00:00Z,2022-06-05T00:00:00Z,1831.0,5.013005,43944,43121,823,1.872838,2021-06-01,NaN,2.0TD,NaN,3.45,3.45,NaN,NaN,NaN,NaN,Bizkaia,Bilbao,48010.0,9820.0
4,00dcd8d903f75e1f22baa037a7a3ab0fd9bc883614a00c...,2017-05-31T01:00:00Z,2022-06-05T00:00:00Z,1831.0,5.013005,43944,43647,297,0.675860,2021-06-01,NaN,2.0TD,NaN,3.30,3.30,NaN,NaN,NaN,NaN,Bizkaia,Bilbao,48006.0,9820.0


Funciones de carga, limpieza e imputación:

In [ ]:
def get_user_path(user_id: str) -> Path:
    return RAW_DIR / f"{user_id}.csv"

def load_raw_user(user_id: str):
    """Carga un CSV sin cabecera y devuelve la serie antes de reindexar."""

    path = get_user_path(user_id)

    if not path.exists():
        return None, {
            "user_id": user_id,
            "status": "no_file",
            "error": "",
            "file": str(path),
        }

    raw = pd.read_csv(
        path,
        header=None,
        names=RAW_COLUMN_NAMES,
        usecols=[0, 1],
        sep=RAW_SEPARATOR,
    )

    rows_raw = len(raw)

    raw["Fecha"] = pd.to_datetime(raw["Fecha"], errors="coerce")
    raw["consumo"] = pd.to_numeric(raw["consumo"], errors="coerce")

    invalid_date = int(raw["Fecha"].isna().sum())
    invalid_consumption = int(raw["consumo"].isna().sum())

    raw = raw.dropna(subset=["Fecha"]).copy()

    negative_values = int((raw["consumo"] < 0).sum())

    if TREAT_NEGATIVE_AS_MISSING:
        raw.loc[raw["consumo"] < 0, "consumo"] = np.nan

    duplicate_mask = raw.duplicated(subset=["Fecha"], keep=False)
    duplicate_rows = raw.loc[duplicate_mask]

    n_duplicate_rows = int(duplicate_mask.sum())
    n_duplicate_timestamps = (
        int(duplicate_rows["Fecha"].nunique())
        if n_duplicate_rows
        else 0
    )

    if n_duplicate_rows:
        n_duplicate_conflicts = int(
            (
                duplicate_rows
                .groupby("Fecha")["consumo"]
                .nunique(dropna=False)
                > 1
            ).sum()
        )
    else:
        n_duplicate_conflicts = 0

    raw = (
        raw.groupby("Fecha", as_index=False, sort=True)["consumo"]
           .mean()
           .sort_values("Fecha")
           .reset_index(drop=True)
    )

    quality_base = {
        "user_id": user_id,
        "status": "loaded",
        "error": "",
        "file": str(path),
        "rows_raw": rows_raw,
        "invalid_date_rows": invalid_date,
        "invalid_consumption_rows": invalid_consumption,
        "negative_values": negative_values,
        "duplicate_rows": n_duplicate_rows,
        "duplicate_timestamps": n_duplicate_timestamps,
        "duplicate_conflicts": n_duplicate_conflicts
    }

    return raw, quality_base

def causal_impute_hourly(df: pd.DataFrame):
    """
    Reindexa a frecuencia horaria e imputa usando únicamente datos
    originales anteriores del mismo usuario.

    Orden:
      1) t-168h
      2) t-24h
      3) mediana histórica weekday+hour
      4) mediana histórica hour
      5) mediana histórica global

    Las imputaciones NO alimentan otras imputaciones.
    """

    if df.empty:
        return df.copy(), {}

    start_date = df["Fecha"].min()
    end_date = df["Fecha"].max()

    full_index = pd.date_range(
        start=start_date,
        end=end_date,
        freq=EXPECTED_FREQ,
    )

    hourly = (
        df.set_index("Fecha")
          .reindex(full_index)
          .rename_axis("Fecha")
          .reset_index()
    )

    original = hourly["consumo"].copy()

    missing_before = original.isna()
    n_missing_before = int(missing_before.sum())

    dow = hourly["Fecha"].dt.dayofweek
    hour = hourly["Fecha"].dt.hour

    lag_168 = original.shift(168)
    lag_24 = original.shift(24)

    aux = pd.DataFrame({
        "_dow": dow,
        "_hour": hour,
        "_original": original,
    })

    hist_dow_hour_median = (
        aux.groupby(["_dow", "_hour"], sort=False)["_original"]
           .transform(
               lambda s: s.shift(1).expanding(min_periods=1).median()
           )
    )

    hist_hour_median = (
        aux.groupby("_hour", sort=False)["_original"]
           .transform(
               lambda s: s.shift(1).expanding(min_periods=1).median()
           )
    )

    hist_global_median = (
        original.shift(1)
                .expanding(min_periods=1)
                .median()
    )

    filled = original.copy()

    method = pd.Series(
        "observed",
        index=hourly.index,
        dtype="object",
    )

    candidates = [
        ("lag_168h", lag_168),
        ("lag_24h", lag_24),
        ("historical_dow_hour_median", hist_dow_hour_median),
        ("historical_hour_median", hist_hour_median),
        ("historical_global_median", hist_global_median)]

    for method_name, candidate in candidates:
        mask = filled.isna() & candidate.notna()
        filled.loc[mask] = candidate.loc[mask]
        method.loc[mask] = method_name

    still_missing = filled.isna()
    method.loc[still_missing] = "not_imputed"

    hourly["consumo"] = filled.astype("float32")
    hourly["imputed"] = missing_before.astype("uint8")
    hourly["imputation_method"] = method.astype(str)

    method_counts = (
        hourly.loc[hourly["imputed"] == 1, "imputation_method"]
        .value_counts()
        .to_dict()
    )

    imputation_info = {
        "start_date": start_date,
        "end_date": end_date,
        "expected_rows_hourly": len(hourly),
        "missing_hours_or_values_before_imputation": n_missing_before,
        "imputed_rows": int(
            ((hourly["imputed"] == 1) & hourly["consumo"].notna()).sum()
        ),
        "remaining_nan_after_imputation": int(hourly["consumo"].isna().sum()),
        "imputed_pct": (
            100.0 * n_missing_before / len(hourly)
            if len(hourly)
            else np.nan
        ),
        "imputation_methods": method_counts}

    return hourly, imputation_info

def build_user_series(user_id: str):
    """Pipeline completo para un usuario."""

    raw, quality = load_raw_user(user_id)

    if raw is None:
        return None, quality

    if raw.empty:
        quality.update({
            "status": "empty",
            "start_date": pd.NaT,
            "end_date": pd.NaT,
            "rows_final": 0,
        })
        return None, quality

    hourly, imp_info = causal_impute_hourly(raw)

    hourly.insert(0, "user_id", str(user_id))

    hourly = hourly[
        [
            "user_id",
            "Fecha",
            "consumo",
            "imputed",
            "imputation_method",
        ]
    ]

    quality.update(imp_info)
    quality["status"] = "ok"
    quality["rows_final"] = len(hourly)

    method_counts = imp_info.get("imputation_methods", {})

    quality["imputed_lag_168h"] = int(method_counts.get("lag_168h", 0))
    quality["imputed_lag_24h"] = int(method_counts.get("lag_24h", 0))
    quality["imputed_hist_dow_hour"] = int(
        method_counts.get("historical_dow_hour_median", 0)
    )
    quality["imputed_hist_hour"] = int(
        method_counts.get("historical_hour_median", 0)
    )
    quality["imputed_hist_global"] = int(
        method_counts.get("historical_global_median", 0)
    )
    quality["not_imputed"] = int(
        method_counts.get("not_imputed", 0)
    )

    quality.pop("imputation_methods", None)

    return hourly, quality

Construir el Parquet con todos los usuarios seleccionados:

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tmp_parquet = OUTPUT_PARQUET.with_suffix(".tmp.parquet")

if OUTPUT_PARQUET.exists():
    if OVERWRITE:
        OUTPUT_PARQUET.unlink()
    else:
        raise FileExistsError(
            f"Ya existe {OUTPUT_PARQUET}."
        )

if tmp_parquet.exists():
    tmp_parquet.unlink()

arrow_schema = pa.schema([
    pa.field("user_id", pa.string()),
    pa.field("Fecha", pa.timestamp("ns")),
    pa.field("consumo", pa.float32()),
    pa.field("imputed", pa.uint8()),
    pa.field("imputation_method", pa.string())
])

quality_records = []
writer = None
n_users_written = 0
n_rows_written = 0
n_imputed_written = 0

try:
    writer = pq.ParquetWriter(
        tmp_parquet,
        schema=arrow_schema,
        compression=PARQUET_COMPRESSION,
        use_dictionary=["user_id", "imputation_method"],
        write_statistics=True
    )

    for user_id in tqdm(user_ids, desc="Procesando usuarios"):
        try:
            user_df, quality = build_user_series(user_id)
            quality_records.append(quality)

            if user_df is None or user_df.empty:
                continue

            table = pa.Table.from_pandas(
                user_df,
                schema=arrow_schema,
                preserve_index=False,
                safe=True
            )

            writer.write_table(table)

            n_users_written += 1
            n_rows_written += len(user_df)
            n_imputed_written += int(user_df["imputed"].sum())

        except Exception as exc:
            quality_records.append({
                "user_id": user_id,
                "status": "error",
                "error": f"{type(exc).__name__}: {exc}"
            })

            if not CONTINUE_ON_USER_ERROR:
                raise

finally:
    if writer is not None:
        writer.close()

quality_df = pd.DataFrame(quality_records)

quality_df.to_csv(
    QUALITY_CSV,
    index=False,
)

quality_df.to_parquet(
    QUALITY_PARQUET,
    index=False,
    compression=PARQUET_COMPRESSION,
)

if n_rows_written == 0:
    if tmp_parquet.exists():
        tmp_parquet.unlink()

    raise RuntimeError(
        "No se ha escrito ninguna observación. Revisa METADATA_PATH, RAW_DIR y que los CSV se llamen <user_id>.csv."
    )

os.replace(tmp_parquet, OUTPUT_PARQUET)

print("Proceso terminado.")
print(f"Usuarios escritos: {n_users_written:,} / {len(user_ids):,}")
print(f"Filas totales: {n_rows_written:,}")
print(f"Filas imputadas: {n_imputed_written:,}")
print(
    f"Porcentaje imputado global: "
    f"{100 * n_imputed_written / n_rows_written:.4f}%"
)
print(f"\nParquet de consumos:\n{OUTPUT_PARQUET}")
print(f"\nInforme de calidad:\n{QUALITY_CSV}")

Procesando usuarios: 100%|██████████| 1025/1025 [06:09<00:00,  2.77it/s]

Proceso terminado.
Usuarios escritos: 1,025 / 1,025
Filas totales: 41,169,408
Filas imputadas: 317,226
Porcentaje imputado global: 0.7705%

Parquet de consumos:
C:\Mis documentos\Master\TFM\processed\consumption_all_users.parquet

Informe de calidad:
C:\Mis documentos\Master\TFM\processed\quality_report.csv


Informe de calidad:

In [ ]:
quality_df = pd.read_parquet(QUALITY_PARQUET)

print("Estado de los usuarios:")
display(quality_df["status"].value_counts(dropna=False).rename_axis("status").to_frame("users"))

ok = quality_df.loc[quality_df["status"] == "ok"].copy()

if not ok.empty:
    cols = [
        "rows_raw",
        "rows_final",
        "invalid_date_rows",
        "invalid_consumption_rows",
        "negative_values",
        "duplicate_timestamps",
        "duplicate_conflicts",
        "missing_hours_or_values_before_imputation",
        "remaining_nan_after_imputation",
        "imputed_pct",
    ]

    cols = [c for c in cols if c in ok.columns]

    print("\nResumen:")
    display(ok[cols].describe().T)

    print("\nUsuarios con mayor porcentaje de imputación:")
    display(
        ok.sort_values("imputed_pct", ascending=False)[
            [
                "user_id",
                "start_date",
                "end_date",
                "rows_raw",
                "rows_final",
                "missing_hours_or_values_before_imputation",
                "imputed_pct",
                "remaining_nan_after_imputation",
            ]
        ].head(30)
    )

    print("\nUso de cada método de imputación:")
    method_columns = [
        "imputed_lag_168h",
        "imputed_lag_24h",
        "imputed_hist_dow_hour",
        "imputed_hist_hour",
        "imputed_hist_global",
        "not_imputed",
    ]

    display(
        ok[method_columns]
        .sum()
        .rename("rows")
        .to_frame()
    )

problems = quality_df.loc[
    quality_df["status"].isin(["error", "no_file", "empty"])
]

if not problems.empty:
    print("\nUsuarios con problemas:")
    display(
        problems[
            [c for c in ["user_id", "status", "error", "file"] if c in problems.columns]
        ].head(100)
    )

Estado de los usuarios:


,users
status,
ok,1025



Resumen:


,count,mean,std,min,25%,50%,75%,max
rows_raw,1025.0,39855.787317,5601.729074,13388.000000,36496.000000,43575.000000,43840.000000,43937.0000
rows_final,1025.0,40165.276098,5358.687454,26424.000000,36649.000000,43920.000000,43944.000000,44016.0000
invalid_date_rows,1025.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
invalid_consumption_rows,1025.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
negative_values,1025.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
duplicate_timestamps,1025.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
duplicate_conflicts,1025.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
missing_hours_or_values_before_imputation,1025.0,309.488780,1736.913526,5.000000,9.000000,57.000000,118.000000,26247.0000
remaining_nan_after_imputation,1025.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
imputed_pct,1025.0,0.771284,4.433269,0.013528,0.023524,0.129711,0.273075,63.9173



Usuarios con mayor porcentaje de imputación:


,user_id,start_date,end_date,rows_raw,rows_final,missing_hours_or_values_before_imputation,imputed_pct,remaining_nan_after_imputation
402,66389429a60faabf5a5d8b73a25c0b6d37dcecb4e333f7...,2017-05-31 01:00:00,2022-02-05,14817,41064,26247,63.917300,0
953,efe8d8a327e8df89b5906904f6dc2cd7e4a57806a0d477...,2018-06-05 01:00:00,2022-05-05,13388,34320,20932,60.990676,0
276,461c48211e7d53887ce74419eba30f3565f333fcfd45d3...,2017-05-31 01:00:00,2022-06-05,19656,43944,24288,55.270344,0
978,f471c8f9626d84a950e4774478ddfee413e6a4f0728064...,2017-05-31 01:00:00,2022-06-05,22507,43944,21437,48.782541,0
943,ee23472b308a6536b8ec5928be20ea08ed89f75348e1cf...,2017-06-02 01:00:00,2021-04-16,18902,33936,15034,44.301037,0
1021,fe73e986873777ffb1973d9ab36485992bebe057ae6bbe...,2018-07-01 01:00:00,2022-05-04,19627,33672,14045,41.711214,0
866,dadd1a1228311311a3937be77c2f623040766ef55b80c7...,2017-10-03 01:00:00,2022-03-18,27188,39048,11860,30.372874,0
548,8b873da53b135788e0d6b7f2aed92b9537beb65e2963d3...,2017-05-31 01:00:00,2022-06-05,33858,43944,10086,22.951939,0
636,a2ada1b5df8539f6479d5f0f6c5bf96ff4318fa4762dca...,2017-05-31 01:00:00,2021-02-09,26538,32400,5862,18.092593,0
712,b896709323b7ac9b55308ddafa52851d1f810963a53e6e...,2017-05-31 01:00:00,2020-11-14,25220,30312,5092,16.798628,0



Uso de cada método de imputación:


,rows
imputed_lag_168h,84812
imputed_lag_24h,7669
imputed_hist_dow_hour,214870
imputed_hist_hour,9833
imputed_hist_global,42
not_imputed,0


Verificar el Parquet:

In [ ]:
parquet_file = pq.ParquetFile(OUTPUT_PARQUET)

print("Esquema:")
print(parquet_file.schema)

print(f"\nNúmero de row groups: {parquet_file.num_row_groups:,}")
print(f"Número total de filas: {parquet_file.metadata.num_rows:,}")
print(f"Tamaño en disco: {OUTPUT_PARQUET.stat().st_size / 1024**3:.3f} GB")

first_group = parquet_file.read_row_group(0).to_pandas()

print("\nPrimer row group:")
display(first_group.head(10))

print("\nTipos:")
print(first_group.dtypes)

print("\nFilas imputadas del ejemplo:")
display(first_group.loc[first_group["imputed"] == 1].head(20))

Esquema:
required group field_id=-1 schema {
  optional binary field_id=-1 user_id (String);
  optional int64 field_id=-1 Fecha (Timestamp(isAdjustedToUTC=false, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional float field_id=-1 consumo;
  optional int32 field_id=-1 imputed (Int(bitWidth=8, isSigned=false));
  optional binary field_id=-1 imputation_method (String);
}


Número de row groups: 1,025
Número total de filas: 41,169,408
Tamaño en disco: 0.361 GB

Primer row group:


,user_id,Fecha,consumo,imputed,imputation_method
0,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 01:00:00,0.064,0,observed
1,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 02:00:00,0.054,0,observed
2,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 03:00:00,0.041,0,observed
3,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 04:00:00,0.055,0,observed
4,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 05:00:00,0.055,0,observed
5,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 06:00:00,0.036,0,observed
6,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 07:00:00,0.064,0,observed
7,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 08:00:00,0.211,0,observed
8,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 09:00:00,0.195,0,observed
9,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-05-31 10:00:00,0.209,0,observed



Tipos:
user_id                         str
Fecha                datetime64[ns]
consumo                     float32
imputed                       uint8
imputation_method               str
dtype: object

Filas imputadas del ejemplo:


,user_id,Fecha,consumo,imputed,imputation_method
48,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 01:00:00,0.070,1,lag_24h
49,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 02:00:00,0.053,1,lag_24h
50,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 03:00:00,0.061,1,lag_24h
51,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 04:00:00,0.060,1,lag_24h
52,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 05:00:00,0.038,1,lag_24h
53,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 06:00:00,0.060,1,lag_24h
54,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 07:00:00,0.181,1,lag_24h
55,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 08:00:00,0.193,1,lag_24h
56,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 09:00:00,0.169,1,lag_24h
57,005932ecd8e0888863372c2ca998801ea6a7f2fef1e9dd...,2017-06-02 10:00:00,0.253,1,lag_24h
